# Pipeline de Recomendación Multimodal de Libros

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 0. Configuración e Importaciones

In [ ]:
#@title Instalación de Librerías y Definición de Constantes
!pip install -q open_clip_torch

import pandas as pd
import numpy as np
import torch
import open_clip
from PIL import Image
import requests
from io import BytesIO

# --- Constantes para OpenCLIP (ejemplo, ajustar según el modelo usado en el catálogo) ---
# Estas deben coincidir con el modelo usado para generar los embeddings del catálogo
# Es crucial que 'model_name_clip' y 'pretrained' sean los mismos.
model_name_clip = "ViT-B-32"
pretrained_model_name = "openai" # Ejemplo, ajustar según el pretrained usado
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Usando dispositivo: {device}")
print(f"Modelo OpenCLIP: {model_name_clip}, Pretrained: {pretrained_model_name}")

# Path al archivo del catálogo en Google Drive (ajustar si es necesario)
catalog_path = "/content/drive/MyDrive/Tesis/data/processed/books_with_embeddings_norm.parquet"
test_image_path = "/content/drive/MyDrive/Tesis/data/test/02_portada_libro_prueba.jpg"

# Columnas esperadas en el DataFrame del catálogo
METADATA_COLS = ['book_id', 'titulo', 'authors', 'image_url'] # Ajustar si es necesario
VISUAL_EMBEDDING_COL = 'normalized_image_embeddings'
TEXT_EMBEDDING_COL = 'normalized_embeddings'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
Usando dispositivo: cpu
Modelo OpenCLIP: ViT-B-32, Pretrained: openai


## 1. Carga de Recursos

In [ ]:
# Implementar la carga del catálogo y sus verificaciones.
# 1. Cargar el dataframe del catálogo desde `catalog_path`.
# 2. Verificar que no hay NaNs ni infinitos en los embeddings visuales y textuales.
# 3. Verificar que los embeddings visuales y textuales tienen norma L2 cercana a 1.

def load_catalog_and_validate(catalog_path, visual_embedding_col, text_embedding_col):
    print(f"Cargando catálogo desde: {catalog_path}")
    catalog_df = pd.read_parquet(catalog_path)
    print(f"Catálogo cargado con {len(catalog_df)} libros.")
    #print(f"Columnas del catalogo cargado :{catalog_df.columns}")

    # Verificar NaNs e Infinitos en embeddings visuales
    if catalog_df[visual_embedding_col].apply(lambda x: np.isnan(x).any() or np.isinf(x).any()).any():
        raise ValueError(f"Embeddings visuales en '{visual_embedding_col}' contienen NaNs o Infinitos.")
    print(f"Verificación de NaNs/Infinitos en embeddings visuales completa.")

    # Verificar NaNs e Infinitos en embeddings textuales
    if catalog_df[text_embedding_col].apply(lambda x: np.isnan(x).any() or np.isinf(x).any()).any():
        raise ValueError(f"Embeddings textuales en '{text_embedding_col}' contienen NaNs o Infinitos.")
    print(f"Verificación de NaNs/Infinitos en embeddings textuales completa.")

    # Verificar norma L2 de embeddings visuales
    visual_norms = catalog_df[visual_embedding_col].apply(lambda x: np.linalg.norm(x))
    if not np.allclose(visual_norms, 1.0, atol=1e-5):
        raise ValueError(f"No todos los embeddings visuales en '{visual_embedding_col}' tienen norma L2 cercana a 1.")
    print(f"Verificación de norma L2 en embeddings visuales completa.")

    # Verificar norma L2 de embeddings textuales
    text_norms = catalog_df[text_embedding_col].apply(lambda x: np.linalg.norm(x))
    if not np.allclose(text_norms, 1.0, atol=1e-5):
        raise ValueError(f"No todos los embeddings textuales en '{text_embedding_col}' tienen norma L2 cercana a 1.")
    print(f"Verificación de norma L2 en embeddings textuales completa.")

    print("Todas las verificaciones de embeddings completadas exitosamente.")
    return catalog_df

try:
    catalog_df = load_catalog_and_validate(catalog_path, VISUAL_EMBEDDING_COL, TEXT_EMBEDDING_COL)
except ValueError as e:
    print(f"Error al cargar o validar el catálogo: {e}")
    catalog_df = None # Asegurarse de que catalog_df no esté definido si hay un error

Cargando catálogo desde: /content/drive/MyDrive/Tesis/data/processed/books_with_embeddings_norm.parquet
Catálogo cargado con 6585 libros.
Verificación de NaNs/Infinitos en embeddings visuales completa.
Verificación de NaNs/Infinitos en embeddings textuales completa.
Verificación de norma L2 en embeddings visuales completa.
Verificación de norma L2 en embeddings textuales completa.
Todas las verificaciones de embeddings completadas exitosamente.


## 2. Carga del Modelo OpenCLIP y Funciones de Encoding

In [ ]:
# Implementar la carga del modelo OpenCLIP y la función `encode_query_image`.
# 1. Usar `open_clip.create_model_and_transforms`.
# 2. Poner el modelo en modo evaluación (`model.eval()`).
# 3. Utilizar el preprocesamiento oficial de OpenCLIP.
# 4. Definir `encode_query_image(image_input)` que acepte path local o URL y devuelva un embedding visual L2 normalizado.
# 5. Definir `encode_query_text(text_input)` que acepte una cadena de texto y devuelva un embedding textual L2 normalizado.

from sklearn.preprocessing import normalize # Importar para usar la normalización de scikit-learn

# Carga del modelo OpenCLIP
model_clip, _, preprocess = open_clip.create_model_and_transforms(model_name_clip, pretrained=pretrained_model_name, device=device)
model_clip.eval() # Poner el modelo en modo evaluación

def encode_query_image(image_input):
    if isinstance(image_input, str):
        if image_input.startswith(('http://', 'https://')):
            response = requests.get(image_input)
            image = Image.open(BytesIO(response.content)).convert("RGB")
        else:
            image = Image.open(image_input).convert("RGB")
    elif isinstance(image_input, Image.Image):
        image = image_input.convert("RGB")
    else:
        raise TypeError("image_input debe ser una URL, un path de archivo o un objeto PIL.Image")

    # Preprocesar la imagen y moverla al dispositivo
    image_preprocessed = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        image_features = model_clip.encode_image(image_preprocessed)
        # Convertir a numpy y normalizar con scikit-learn
        normalized_image_features = normalize(image_features.cpu().numpy(), axis=1, norm='l2')

    return normalized_image_features.flatten()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


## 3. Recomendador Visual Baseline

In [ ]:
# Implementar la función `recommend_visual_only(query_image, visual_embeddings, metadata, k=10)`.
# 1. Generar el embedding visual de la imagen de consulta.
# 2. Calcular scores visuales (producto punto).
# 3. Obtener los top-k resultados en orden descendente.
# 4. Devolver un DataFrame con las columnas especificadas.

def recommend_visual_only(query_image_input, catalog_dataframe, k=10):
    # 1. Generar el embedding visual de la imagen de consulta.
    query_embedding = encode_query_image(query_image_input)

    # Extraer los embeddings visuales y la metadata del DataFrame del catálogo
    visual_embeddings = np.stack(catalog_dataframe[VISUAL_EMBEDDING_COL].values)
    metadata_df = catalog_dataframe[METADATA_COLS]

    # Asegurar que el embedding de la consulta tenga la forma correcta para el producto punto
    # query_embedding tiene forma (embedding_dim,) después de flatten()
    # visual_embeddings debe ser (num_books, embedding_dim)

    # 2. Calcular scores visuales (producto punto). Como ambos están normalizados, es similitud coseno.
    # Expandir la dimensión de query_embedding para que sea (1, embedding_dim) para el producto matricial
    visual_scores = np.dot(query_embedding, visual_embeddings.T)

    # 3. Obtener los top-k resultados en orden descendente.
    top_k_indices = np.argsort(visual_scores)[::-1][:k]

    # Crear DataFrame de resultados
    results_df = metadata_df.iloc[top_k_indices].copy()
    results_df['visual_score'] = visual_scores[top_k_indices]

    return results_df.reset_index(drop=True)

## 5. Recomendador Multimodal Asimétrico

In [ ]:
# Implementar la función `recommend_multimodal(query_image, visual_embeddings, text_embeddings, metadata, k=10, m=50, alpha=0.7)`.
# 1. Generar el embedding visual de la consulta.
# 2. Calcular scores visuales contra todo el catálogo.
# 3. Seleccionar top-M candidatos visuales.
# 4. Tomar el top-1 visual como ancla textual.
# 5. Calcular `text_scores`.
# 6. Fusionar scores visuales y textuales.
# 7. Ordenar por scores finales.
# 8. Devolver un DataFrame con las columnas especificadas y las validaciones de `alpha` y `m`.

def recommend_multimodal(query_image_input, catalog_dataframe, k=10, m=50, alpha=0.7):
    # Validaciones iniciales de parámetros
    if not (0 <= alpha <= 1):
        raise ValueError("Alpha debe estar entre 0 y 1.")
    if not (1 <= k <= m):
        raise ValueError("k debe ser menor o igual que m, y ambos deben ser mayores o iguales a 1.")

    # 1. Generar el embedding visual de la consulta.
    query_visual_embedding = encode_query_image(query_image_input)

    # Extraer los embeddings del catálogo y la metadata
    catalog_visual_embeddings = np.stack(catalog_dataframe[VISUAL_EMBEDDING_COL].values)
    catalog_text_embeddings = np.stack(catalog_dataframe[TEXT_EMBEDDING_COL].values)
    metadata_df = catalog_dataframe[METADATA_COLS]

    # 2. Calcular scores visuales contra todo el catálogo (similitud coseno).
    visual_scores_all = np.dot(query_visual_embedding, catalog_visual_embeddings.T)

    # 3. Seleccionar top-M candidatos visuales (índices del catálogo completo).
    top_m_visual_indices_all = np.argsort(visual_scores_all)[::-1][:m]

    # 4. Tomar el top-1 visual como ancla textual.
    # Este es el embedding textual del libro que es visualmente más similar a la consulta.
    top_1_visual_index = top_m_visual_indices_all[0]
    textual_anchor_embedding = catalog_text_embeddings[top_1_visual_index]

    # 5. Calcular `text_scores` utilizando el ancla textual contra *todo* el catálogo textual.
    text_scores_all = np.dot(textual_anchor_embedding, catalog_text_embeddings.T)

    # Excluir la auto-similitud del ancla textual, evitando que obtenga un score de 1.0 por sí mismo.
    # Esto asegura que el libro ancla no sea el top resultado únicamente por su score textual perfecto.
    text_scores_all[top_1_visual_index] = -1.0  # Establecer un valor bajo para penalizar la auto-similitud

    # Obtener los scores visuales y textuales para los M candidatos
    visual_scores_m_candidates = visual_scores_all[top_m_visual_indices_all]
    text_scores_m_candidates = text_scores_all[top_m_visual_indices_all]

    # 6. Fusionar scores visuales y textuales para los M candidatos.
    final_scores_m_candidates = alpha * visual_scores_m_candidates + (1 - alpha) * text_scores_m_candidates

    # 7. Ordenar por scores finales (de los M candidatos) y obtener los top-k.
    top_k_in_m_indices = np.argsort(final_scores_m_candidates)[::-1][:k]

    # Mapear estos índices de vuelta a los índices originales del catálogo
    final_top_k_catalog_indices = top_m_visual_indices_all[top_k_in_m_indices]

    # 8. Devolver un DataFrame con las columnas especificadas y los scores.
    results_df = metadata_df.iloc[final_top_k_catalog_indices].copy()
    results_df['visual_score'] = visual_scores_all[final_top_k_catalog_indices]
    results_df['text_score'] = text_scores_all[final_top_k_catalog_indices]
    results_df['final_multimodal_score'] = final_scores_m_candidates[top_k_in_m_indices]

    return results_df.reset_index(drop=True)

In [ ]:
# Implementar la función `recommend_multimodal_top_n(query_image, catalog_dataframe, k=10, m=50, alpha=0.7, n_top_visual=5)`.
# Esta función es una variación de `recommend_multimodal` donde el ancla textual
# se calcula promediando los embeddings textuales de los `n_top_visual` libros
# visualmente más similares a la imagen de consulta.

def recommend_multimodal_top_n(query_image_input, catalog_dataframe, k=10, m=50, alpha=0.7, n_top_visual=5):
    # Validaciones iniciales de parámetros
    if not (0 <= alpha <= 1):
        raise ValueError("Alpha debe estar entre 0 y 1.")
    if not (1 <= k <= m):
        raise ValueError("k debe ser menor o igual que m, y ambos deben ser mayores o iguales a 1.")
    if not (1 <= n_top_visual <= m):
        raise ValueError("n_top_visual debe ser mayor o igual a 1 y menor o igual que m.")
    # Nueva validación: Asegurar que hay suficientes candidatos en 'm' después de excluir los 'n_top_visual' anchors
    if m < k + n_top_visual:
        raise ValueError(f"'m' ({m}) debe ser mayor o igual que 'k' ({k}) + 'n_top_visual' ({n_top_visual}) "
                         "para garantizar suficientes candidatos después de la exclusión.")

    # 1. Generar el embedding visual de la consulta.
    query_visual_embedding = encode_query_image(query_image_input)

    # Extraer los embeddings del catálogo y la metadata
    # np.stack se usa para convertir la Serie de arrays de numpy en un solo array 2D.
    # Si los embeddings ya fueran matrices globales, se podría evitar este paso.
    catalog_visual_embeddings = np.stack(catalog_dataframe[VISUAL_EMBEDDING_COL].values)
    catalog_text_embeddings = np.stack(catalog_dataframe[TEXT_EMBEDDING_COL].values)
    metadata_df = catalog_dataframe[METADATA_COLS]

    # 2. Calcular scores visuales contra todo el catálogo (similitud coseno).
    visual_scores_all = np.dot(query_visual_embedding, catalog_visual_embeddings.T)

    # 3. Seleccionar top-M candidatos visuales (índices del catálogo completo).
    top_m_visual_indices_all = np.argsort(visual_scores_all)[::-1][:m]

    # 4. Tomar el top-N visual para calcular el ancla textual promedio.
    # Obtener los embeddings textuales de los top N visuales y promediarlos.
    top_n_visual_indices_for_anchor = top_m_visual_indices_all[:n_top_visual]
    blended_textual_anchor_embeddings = catalog_text_embeddings[top_n_visual_indices_for_anchor]
    textual_anchor_embedding = np.mean(blended_textual_anchor_embeddings, axis=0)
    # Normalizar el ancla textual promediada (importante si los embeddings originales están normalizados)
    textual_anchor_embedding = normalize(textual_anchor_embedding.reshape(1, -1), axis=1, norm='l2').flatten()

    # 5. Calcular `text_scores` utilizando el ancla textual blended contra *todo* el catálogo textual.
    text_scores_all = np.dot(textual_anchor_embedding, catalog_text_embeddings.T)

    # Crear un conjunto de los índices de los libros que se usaron como anchors.
    # Estos libros serán excluidos de los resultados finales.
    anchor_indices = set(top_n_visual_indices_for_anchor)

    # Obtener los scores visuales y textuales para los M candidatos iniciales
    visual_scores_m_candidates = visual_scores_all[top_m_visual_indices_all]
    text_scores_m_candidates = text_scores_all[top_m_visual_indices_all]

    # 6. Fusionar scores visuales y textuales para los M candidatos.
    final_scores_m_candidates = alpha * visual_scores_m_candidates + (1 - alpha) * text_scores_m_candidates

    # --- Lógica de exclusión de anchors y selección final de top-k ---
    # Se crea una máscara para identificar qué candidatos entre los 'm' no son anchors.
    is_not_anchor_mask = ~np.isin(top_m_visual_indices_all, list(anchor_indices))

    # Se aplican los filtros a los índices y a los scores de los 'm' candidatos.
    filtered_candidate_catalog_indices = top_m_visual_indices_all[is_not_anchor_mask]
    filtered_visual_scores = visual_scores_m_candidates[is_not_anchor_mask]
    filtered_text_scores = text_scores_m_candidates[is_not_anchor_mask]
    filtered_final_scores = final_scores_m_candidates[is_not_anchor_mask]

    # 7. Ordenar por scores finales (de los candidatos filtrados) y obtener los top-k.
    # Se obtienen los índices dentro de la lista de candidatos filtrados.
    top_k_in_filtered_indices = np.argsort(filtered_final_scores)[::-1][:k]

    # Mapear estos índices de vuelta a los índices originales del catálogo para los resultados finales.
    final_top_k_catalog_indices = filtered_candidate_catalog_indices[top_k_in_filtered_indices]

    # 8. Devolver un DataFrame con las columnas especificadas y los scores.
    results_df = metadata_df.iloc[final_top_k_catalog_indices].copy() # Usar .iloc con los índices finales
    results_df['visual_score'] = filtered_visual_scores[top_k_in_filtered_indices] # Scores visuales de los top-k filtrados
    results_df['text_score'] = filtered_text_scores[top_k_in_filtered_indices]     # Scores textuales de los top-k filtrados
    results_df['final_multimodal_score'] = filtered_final_scores[top_k_in_filtered_indices] # Scores multimodales de los top-k filtrados

    return results_df.reset_index(drop=True)


## 6. Visualización de Resultados

In [ ]:
# Implementar la función `show_recommendations(df_results)`.
# 1. Mostrar título, autor, scores e imagen de portada.
# 2. Manejar casos donde `image_url` falle.

## 7. Demostración y Validaciones Adicionales

In [ ]:
# Implementar una demostración del pipeline completo con una imagen de consulta de ejemplo.
# Incluir las validaciones:
# - `q.shape[0] == visual_embeddings.shape[1]`
# - Evitar recomendar el mismo libro si la query proviene de una imagen del catálogo.